# Thursday — Another Way to See: Object Detection with YOLO

Your recognizer is a **classifier**: give it *one cropped object* and it says *which one it is*. Today you'll meet a different kind of vision model — a **detector** — by running the most famous one, **YOLO** ("You Only Look Once").

A detector looks at a **whole scene** and, in a single pass, **finds every object and draws a box around each** — with a label and a confidence. You'll run a pretrained YOLO, watch it work, and compare it to the classifier you built.

> **This is enrichment.** For the robot arm we still use *your* classifier plus the camera's crop — it's lighter, needs far less data, and the depth camera already crops the object for us. YOLO is a great tool to know; it's just not the one the arm needs.

Look for the **🔧 YOUR TURN** cells (`# TODO`, blanks written as `____`).


> **How to run this:** You're on our GPU server through **JupyterHub**. The first cell downloads the YOLO library and a tiny model file, so give it a moment. The optional webcam section at the end runs on **your own laptop** (it needs your camera).

In [ ]:
# Install the YOLO library (one time). If it's already there, this is quick.
# !pip install ultralytics
from ultralytics import YOLO
import matplotlib.pyplot as plt
from collections import Counter
print("ready")


## 1. 🔧 YOUR TURN — load a pretrained YOLO

We'll use the **nano** model, `yolov8n.pt` — the smallest, most edge-friendly YOLO. It was already trained on **COCO**, a dataset of 80 everyday object types (person, cup, chair, laptop, banana…), so it can detect them out of the box.


In [ ]:
# TODO 1: load the nano model. Fill in the filename 'yolov8n.pt'.
model = YOLO("____")

print("YOLO loaded. It knows", len(model.names), "object types, for example:")
print([model.names[i] for i in range(10)])


## 2. 🔧 YOUR TURN — run it on an image

Point it at any photo with objects in it. You can use one of **your own** photos from this week, or the sample below. The model returns a list of results — one entry per image.


In [ ]:
# TODO 2: set IMG to an image path (or the sample URL).
# Try a busy photo (a desk, a room) so there's more to find.
IMG = "____"          # e.g. "my_objects/val/mug/img001.jpg"  or  "https://ultralytics.com/images/bus.jpg"

results = model(IMG)
print("done —", len(results[0].boxes), "objects detected")


## 3. See the boxes

`.plot()` draws every detection on the image (box + label + confidence). Nothing to fill in — just run it.


In [ ]:
annotated = results[0].plot()          # image with boxes drawn (BGR color order)
plt.figure(figsize=(9, 6))
plt.imshow(annotated[:, :, ::-1])       # flip BGR -> RGB so colors look right
plt.axis("off"); plt.title("YOLO detections"); plt.show()


## 4. 🔧 YOUR TURN — what did it find?

Each detection has a **class** (what) and a **confidence** (how sure). Let's list them and count how many of each.


In [ ]:
boxes = results[0].boxes

# TODO 3: how many objects were detected?  hint: len(boxes)
num_detections = ____
print("total detections:", num_detections)

# Turn each class id into its name and count them:
names = [model.names[int(c)] for c in boxes.cls]
print("counts by object:", dict(Counter(names)))


## 5. 🔧 YOUR TURN — confidence threshold

Just like your classifier, a detector gives a **confidence** for every guess. Low-confidence boxes are often junk. Keep only the ones the model is sure about.


In [ ]:
THRESH = 0.5   # try 0.25, 0.5, 0.8 and see how the count changes

confident = [float(c) for c in boxes.conf if float(c) >= THRESH]

# TODO 4: how many detections are at or above THRESH?  hint: len(confident)
num_confident = ____
print(f"{num_confident} of {len(boxes)} detections are >= {THRESH:.0%} confident")


## 6. Detector vs. classifier — the big picture

| | **Your classifier** | **YOLO (detector)** |
|---|---|---|
| Input | one cropped object | a whole scene |
| Output | one label | many boxes + labels |
| Answers | *what is this?* | *what is where?* |
| Data to train | ~50 photos per class | thousands of hand-drawn boxes |

**So when would the arm want a detector?** If objects weren't pre-cropped, or if it had to handle a messy pile of many objects at once. But our arm's depth camera **already crops** one object and hands it over — so a lightweight classifier is the perfect, cheap fit. Same reason phones use small models: pick the lightest tool that does the job.


## 7. (optional) Live detection on your laptop webcam

Run this **on your own laptop** (not the server — it has no camera). A window opens showing live detections; close it or press **q** to stop.


In [ ]:
# Laptop only. source=0 is your default webcam.
# model.predict(source=0, show=True)

# No webcam? Point it at a folder of images instead:
# model.predict(source="my_objects/val/mug", show=False, save=True)   # saves annotated copies


## 8. Bringing it back to today: YOLO on the edge

YOLO plays by the **same rules you learned this morning**. `yolov8n` is already tiny, and you can **export it to ONNX and quantize it** — exactly what you did to your recognizer. Try the challenge below and compare the file sizes.


In [ ]:
# Export YOLO to ONNX (writes yolov8n.onnx):
# path = model.export(format="onnx")
# import os; print("ONNX size:", round(os.path.getsize(path)/1e6, 1), "MB")
# From here you could quantize it with onnxruntime, just like in thursday_optimize.ipynb.


## Challenges (if you finish early)

1. **Go bigger.** Swap `yolov8n.pt` for `yolov8s.pt` (small). Does it find more? Is it slower?
2. **Filter to one thing.** Show only detections of a chosen class (e.g. only `person` or only `cup`).
3. **Measure speed.** Time 20 runs on one image and compute FPS — the same benchmarking idea as this morning.
4. **Try YOUR objects.** Run YOLO on your Day-1 photos. Does COCO already know your objects? Where does it fail? (This is exactly why you trained your *own* classifier.)
5. **Export + shrink.** Export to ONNX (Section 8), then quantize it with `onnxruntime` and compare size/speed to the original.
